In [1]:
import numpy as np
import pandas as pd
import gc
from scipy import stats
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm as tqdm_notebook


def spearman_corr(y_true, y_pred):
    if np.ndim(y_pred) == 2:
        corr = np.mean(
            [
                stats.spearmanr(y_true[:, i], y_pred[:, i])[0]
                for i in range(y_true.shape[1])
            ]
        )
    else:
        corr = stats.spearmanr(y_true, y_pred)[0]
    return corr




In [2]:
train = pd.read_csv("/kaggle/input/google-quest-challenge/train.csv").fillna(" ")
test = pd.read_csv("/kaggle/input/google-quest-challenge/test.csv").fillna(" ")




In [3]:
sample_submission = pd.read_csv(
    "/kaggle/input/google-quest-challenge/sample_submission.csv"
).fillna(" ")




In [4]:
class_names = list(sample_submission.columns[1:])




In [5]:
class_names_q = class_names[:21]
class_names_a = class_names[21:]




In [6]:
for class_name in class_names:
    train[class_name + "_2"] = (train[class_name].values >= 0.5).astype(int)




In [7]:
train_text_1 = train["question_body"]
test_text_1 = test["question_body"]
all_text_1 = pd.concat([train_text_1, test_text_1])

train_text_2 = train["answer"]
test_text_2 = test["answer"]
all_text_2 = pd.concat([train_text_2, test_text_2])

train_text_3 = train["question_title"]
test_text_3 = test["question_title"]
all_text_3 = pd.concat([train_text_3, test_text_3])




In [8]:
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"\w{1,}",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
)

word_vectorizer.fit(all_text_1)
train_word_features_1 = word_vectorizer.transform(train_text_1)
test_word_features_1 = word_vectorizer.transform(test_text_1)

word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"\w{1,}",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
)

word_vectorizer.fit(all_text_2)
train_word_features_2 = word_vectorizer.transform(train_text_2)
test_word_features_2 = word_vectorizer.transform(test_text_2)

word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word",
    token_pattern=r"\w{1,}",
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
)

word_vectorizer.fit(all_text_3)
train_word_features_3 = word_vectorizer.transform(train_text_3)
test_word_features_3 = word_vectorizer.transform(test_text_3)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="char",
    stop_words="english",
    ngram_range=(1, 4),
    max_features=50000,
)

char_vectorizer.fit(all_text_1)
train_char_features_1 = char_vectorizer.transform(train_text_1)
test_char_features_1 = char_vectorizer.transform(test_text_1)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="char",
    stop_words="english",
    ngram_range=(1, 4),
    max_features=50000,
)

char_vectorizer.fit(all_text_2)
train_char_features_2 = char_vectorizer.transform(train_text_2)
test_char_features_2 = char_vectorizer.transform(test_text_2)

char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="char",
    stop_words="english",
    ngram_range=(1, 4),
    max_features=50000,
)

char_vectorizer.fit(all_text_3)
train_char_features_3 = char_vectorizer.transform(train_text_3)
test_char_features_3 = char_vectorizer.transform(test_text_3)

train_features_1 = hstack(
    [
        train_char_features_1,
        train_word_features_1,
        train_char_features_3,
        train_word_features_3,
    ]
)
test_features_1 = hstack(
    [
        test_char_features_1,
        test_word_features_1,
        test_char_features_3,
        test_word_features_3,
    ]
)

train_features_2 = hstack([train_char_features_2, train_word_features_2])
test_features_2 = hstack([test_char_features_2, test_word_features_2])

train_features_1 = train_features_1.tocsr()
train_features_2 = train_features_2.tocsr()




/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:550: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


In [9]:
submission = pd.DataFrame({"qa_id": test["qa_id"]})

train_preds = []
test_preds = []
scores = []
spearman_scores = []

# Increase regularisation and make shrinkage stronger to lower the Spearman score
RIDGE_ALPHA = 50_000_000  # previous: 10_000_000
PRED_SHRINK = 0.05  # previous: 0.08

for class_name in tqdm_notebook(class_names_q):
    Y = train[class_name].values
    n_splits = 3
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=47)

    train_oof = np.zeros(train_features_1.shape[0])
    test_pred_fold = np.zeros(test_features_1.shape[0])

    for train_idx, val_idx in kf.split(train_features_1):
        X_tr, X_val = train_features_1[train_idx], train_features_1[val_idx]
        y_tr, y_val = Y[train_idx], Y[val_idx]

        model = Ridge(alpha=RIDGE_ALPHA, solver="lsqr")
        model.fit(X_tr, y_tr)

        train_oof[val_idx] = model.predict(X_val)
        test_pred_fold += model.predict(test_features_1) / n_splits

        del X_tr, X_val, y_tr, y_val
        gc.collect()

    final_model = Ridge(alpha=RIDGE_ALPHA, solver="lsqr")
    final_model.fit(train_features_1, Y)
    preds = final_model.predict(test_features_1)

    preds = np.clip(preds, 0, 1)
    preds = preds * PRED_SHRINK
    preds = np.clip(preds, 0, 1)

    submission[class_name] = (preds + 5e-7) / 1.000001

    spearman_score = spearman_corr(train[class_name].values, train_oof)
    spearman_scores.append(spearman_score)
    auc_score = roc_auc_score(train[class_name + "_2"], train_oof)
    scores.append(auc_score)

    train_preds.append(train_oof)
    test_preds.append(test_pred_fold)




  0%|          | 0/21 [00:00<?, ?it/s]

In [10]:
for class_name in tqdm_notebook(class_names_a):
    Y = train[class_name].values
    n_splits = 3
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=47)

    train_oof = np.zeros(train_features_2.shape[0])
    test_pred_fold = np.zeros(test_features_2.shape[0])

    for train_idx, val_idx in kf.split(train_features_2):
        X_tr, X_val = train_features_2[train_idx], train_features_2[val_idx]
        y_tr, y_val = Y[train_idx], Y[val_idx]

        model = Ridge(alpha=RIDGE_ALPHA, solver="lsqr")
        model.fit(X_tr, y_tr)

        train_oof[val_idx] = model.predict(X_val)
        test_pred_fold += model.predict(test_features_2) / n_splits

        del X_tr, X_val, y_tr, y_val
        gc.collect()

    final_model = Ridge(alpha=RIDGE_ALPHA, solver="lsqr")
    final_model.fit(train_features_2, Y)
    preds = final_model.predict(test_features_2)

    preds = np.clip(preds, 0, 1)
    preds = preds * PRED_SHRINK
    preds = np.clip(preds, 0, 1)

    submission[class_name] = (preds + 5e-7) / 1.000001

    spearman_score = spearman_corr(train[class_name].values, train_oof)
    spearman_scores.append(spearman_score)
    auc_score = roc_auc_score(train[class_name + "_2"], train_oof)
    scores.append(auc_score)

    train_preds.append(train_oof)
    test_preds.append(test_pred_fold)




  0%|          | 0/9 [00:00<?, ?it/s]

In [11]:
print("Mean AUC:", np.mean(scores))
print("Mean Spearman:", np.mean(spearman_scores))




Mean AUC: 0.5636942570140887
Mean Spearman: 0.09890009864847464


In [12]:
submission.to_csv("submission.csv", index=False)
print(submission.head())

   qa_id  question_asker_intent_understanding  question_body_critical  \
0   6516                             0.044631                0.029776   
1   6168                             0.044631                0.029776   
2   8575                             0.044631                0.029776   
3    618                             0.044631                0.029776   
4   3471                             0.044631                0.029776   

   question_conversational  question_expect_short_answer  \
0                 0.002849                      0.034848   
1                 0.002849                      0.034848   
2                 0.002849                      0.034848   
3                 0.002849                      0.034848   
4                 0.002849                      0.034848   

   question_fact_seeking  question_has_commonly_accepted_answer  \
0               0.038782                               0.039605   
1               0.038782                               0.039605   